# Guida MLOps: Monitoraggio Reputazione Aziendale

Benvenuto! In questo notebook esploreremo passo-passo la costruzione e il miglioramento di un progetto di Sentiment Analysis, portandolo da un semplice script a un'architettura **MLOps (Machine Learning Operations)**.

## Indice
1. L'Architettura Iniziale
2. Il Problema del Retraining Sincrono
3. La Soluzione: Automazione e CI/CD
4. Il Deploy su HuggingFace
5. Comandi per Avviare il Progetto


## 1. L'Architettura Iniziale

Il progetto originale era diviso in due container Docker:
- **ML_App**: Un'applicazione FastAPI (in Python) che espone il modello di sentiment analysis.
- **Grafana**: Un cruscotto di visualizzazione (dashboard) per leggere i log.

Ogni volta che si interrogava l'endpoint `/predict`, il modello analizzava il testo e salvava il risultato su un database SQLite locale. Grafana poi leggeva questo database per generare i grafici.

## 2. Il Problema del Retraining Sincrono

Il codice originale aveva un grave difetto di progettazione: l'endpoint `/retrain` eseguiva l'addestramento del modello bloccando completamente il server FastAPI. 
L'addestramento (Training) richiede tanta memoria e CPU, e dura molto tempo (minuti od ore). Farlo girare dentro le API in produzione significa che i tuoi utenti non possono più usare l'applicazione finché il modello non ha finito di studiare!

Inoltre, usare il database SQLite delle vecchie predizioni per ri-addestrare il modello è un errore chiamato **Data Drift**. Il modello finirebbe per imparare dai propri stessi errori passati.

## 3. La Soluzione: Automazione e CI/CD (GitHub Actions)

Per risolvere questo problema, abbiamo diviso le responsabilità:
1. **Abbiamo isolato il codice di training** in un nuovo file `scripts/esegui_training.py`.
2. **Abbiamo delegato il training al Cloud**, usando **GitHub Actions**. 
   Ora, ogni domenica notte (o su comando), GitHub avvia un computer virtuale, entra nel tuo Docker, scarica un dataset fresco da Kaggle tramite il tuo `KAGGLE_API_TOKEN`, e allena il modello da zero, senza mai toccare il server di produzione!

## 4. Il Deploy su HuggingFace

Una volta che il modello è stato addestrato dal workflow di GitHub Actions, i pesi del modello (il "cervello" aggiornato) vengono salvati.
Ma come facciamo a distribuirlo? Abbiamo implementato un sistema che utilizza la libreria `huggingface_hub`.

Se hai configurato `HF_TOKEN` e `HF_REPO_ID` nei segreti di GitHub, il modello appena creato viene automaticamente **pushato sul Cloud di HuggingFace**, diventando subito disponibile per essere scaricato dal server di produzione (tramite l'endpoint `/change_model`).


## 5. Comandi per Avviare il Progetto

Ecco i comandi necessari per avviare il progetto da zero sul tuo computer o sul tuo ambiente Codespaces.

In [ ]:
!git clone https://github.com/begosan/Progetto-Monitoraggio-della-reputazione-online-di-un-azienda.git
!cd Progetto-Monitoraggio-della-reputazione-online-di-un-azienda

Se vuoi fare un retraining locale manuale, ricordati di esportare i token:

In [ ]:
!export KAGGLE_API_TOKEN="IL_TUO_TOKEN_QUI"
!export HF_TOKEN="IL_TUO_TOKEN_HF_QUI"
!export HF_REPO_ID="begosan/monitoraggio"
# e poi avvia l'intero sistema con:
!docker-compose up --build